# Challenge 2: Image Classification

## Module imports

In [15]:
import os
import torch
torch.manual_seed(hparams.SEED)
from torch import nn
from torchsummary import summary
from torch.utils.tensorboard import SummaryWriter
import torchvision
from torchvision.transforms import v2 as transforms
from torch.utils.data import TensorDataset, DataLoader
from torchview import draw_graph
import cv2
import copy
import shutil
from itertools import product
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from PIL import Image
import matplotlib.gridspec as gridspec
import numpy as np

import hparams
import image_classifier

if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(hparams.SEED)
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device("cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")


PyTorch version: 2.9.1+cu128
Device: cuda


## Image Loader

In [ ]:
def load_images_from_folder(folder):
    images = []
    valid_extension = '.png'
    idx = 1

    for filename in os.listdir(folder):
        if not filename.lower().endswith(valid_extension):
            continue

        file_path = os.path.join(folder, filename)
        img = cv2.imread(file_path)
        img = (img / 255.0).astype(np.float32)
        img = img[..., ::-1]
        dim = min(img.shape[:-1])
        h, w = img.shape[:2]
        img = img[(h - dim) // 2 : (h + dim) // 2,
                  (w - dim) // 2 : (w + dim) // 2, :]
        img = cv2.resize(img, (hparams.IMAGE_SIZE, hparams.IMAGE_SIZE))
        images.append(img)

        if idx % 20 == 0:
            print(f"Loaded {idx} images ")
        idx = idx + 1

    return np.array(images)

train_images = load_images_from_folder(hparams.TRAIN_DATA_PATH)
print(f"Loaded {len(train_images)} images for training")

# test_images = load_images_from_folder(hparams.TEST_DATA_PATH)
# print(f"Loaded {len(test_images)} images for testing ")

Loaded 20 images 
Loaded 40 images 
Loaded 60 images 
Loaded 80 images 
Loaded 100 images 
Loaded 120 images 
Loaded 140 images 
Loaded 160 images 
Loaded 180 images 
Loaded 200 images 
Loaded 220 images 
Loaded 240 images 
Loaded 260 images 
Loaded 280 images 
Loaded 300 images 
Loaded 320 images 
Loaded 340 images 
Loaded 360 images 
Loaded 380 images 
Loaded 400 images 
Loaded 420 images 
Loaded 440 images 
Loaded 460 images 
Loaded 480 images 
Loaded 500 images 
Loaded 520 images 
Loaded 540 images 
Loaded 560 images 
Loaded 580 images 
Loaded 600 images 
Loaded 620 images 
Loaded 640 images 
Loaded 660 images 
Loaded 680 images 
Loaded 700 images 
Loaded 720 images 
Loaded 740 images 
Loaded 760 images 
Loaded 780 images 
Loaded 800 images 
Loaded 820 images 
Loaded 840 images 
Loaded 860 images 
Loaded 880 images 
Loaded 900 images 
Loaded 920 images 
Loaded 940 images 
Loaded 960 images 
Loaded 980 images 
Loaded 1000 images 
Loaded 1020 images 
Loaded 1040 images 
Loaded 1060 i